In [1]:
import pandas as pd
import polars as pl

## Key Conceptual Differences

Before diving into specific operations, let's understand the fundamental differences in how pandas and Polars approach DataFrame manipulation.

### No Index in Polars

One of the most significant differences is that Polars has no concept of an index, while in pandas, every DataFrame has an index that plays a central role in many operations.

In [ ]:
df_pd = pd.DataFrame({'A': [101, 102, 103], 'B': ["red", "green", "blue"], 'C': [2, 5, 7]})
df_pd.set_index('A', inplace=True)
df_pd

,B,C
A,,
101,red,2
102,green,5
103,blue,7


In [ ]:
df_pd.loc[102]

B    green
C        5
Name: 102, dtype: object

In Polars, all columns are equal - there's no special index:

Access by index value in pandas:

In [ ]:
df_pl = pl.DataFrame({'A': [101, 102, 103], 'B': ["red", "green", "blue"]})
df_pl

A,B
i64,str
101,"""red"""
102,"""green"""
103,"""blue"""


Filter by condition in Polars:

In [ ]:
df_pl.filter(pl.col('A') == 102)

A,B
i64,str
102,"""green"""


### Immutability vs. In-Place Operations

pandas allows and sometimes encourages in-place modifications, while Polars generally follows an immutable pattern:

In [23]:
df_pd = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
df_pd['C'] = df_pd['B'] * 2
df_pd.drop('B', axis=1, inplace=True)
df_pd

,A,C
0,1,8
1,2,10
2,3,12


#### Polars: Operations return new DataFrames

### Expression API vs. Apply/Lambda

Perhaps the most fundamental difference for everyday coding is how transformations are specified. pandas often relies on Python functions (via `.apply()` or lambdas), while Polars emphasizes declarative expressions:

In [24]:
df_pd = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 6, 8]})
df_pd['C'] = df_pd.apply(lambda row: row['A'] * 2 if row['B'] > 5 else row['A'], axis=1)
df_pd

,A,B,C
0,1,4,1
1,2,6,4
2,3,8,6


#### Polars: Using expressions

### Eager vs. Lazy Evaluation

While pandas operates almost exclusively in eager mode (operations execute immediately), Polars offers both eager and lazy modes:

#### Pandas: Always eager

```python
df_pd = pd.read_csv("data.csv")
filtered = df_pd[df_pd['value'] > 0]  # Executes immediately
result = filtered.groupby('category').sum()  # Executes immediately
```

#### Polars: Eager mode

```python
df_pl = pl.read_csv("data.csv")
filtered = df_pl.filter(pl.col('value') > 0)  # Executes immediately
result = filtered.group_by('category').agg(pl.sum('value'))  # Executes immediately
```

#### Polars: Lazy mode

```python
lazy_df = pl.scan_csv("data.csv")
query = (
    lazy_df.filter(pl.col('value') > 0)
    .group_by('category')
    .agg(pl.col('value').sum())
)
# No execution yet - just building a query plan
result = query.collect()  # Now executes with optimization
```

### Type Strictness

Polars is stricter about types and schema consistency:

In [25]:
df_pd = pd.DataFrame({'A': [1, 2, 'three']})
df_pd

,A
0,1
1,2
2,three


#### Polars: Stricter about types

## Common Operations: pandas vs. Polars

Now let's look at side-by-side comparisons of common operations in both libraries.

### Creating DataFrames

In [26]:
df_pd = pd.DataFrame({
    'A': [1, 2, 3],
    'B': ['a', 'b', 'c'],
    'C': [True, False, True]
})
df_pd

,A,B,C
0,1,a,True
1,2,b,False
2,3,c,True


#### Polars

### Selecting Columns

In [27]:
sub_pd = df_pd[['A', 'B']]
sub_pd

,A,B
0,1,a
1,2,b
2,3,c


#### Polars

### Filtering Rows

In [28]:
filtered_pd = df_pd[df_pd['A'] > 1]
filtered_pd

,A,B,C
1,2,b,False
2,3,c,True


#### Polars

### Adding/Modifying Columns

In [29]:
df_pd['D'] = df_pd['A'] * 2
df_pd

,A,B,C,D
0,1,a,True,2
1,2,b,False,4
2,3,c,True,6


#### Polars

In [52]:
df_pl_mod = pl.DataFrame({'A': [1, 2, 3], 'B': ['a', 'b', 'c'], 'C': [True, False, True]})
df_pl_mod = df_pl_mod.with_columns([
    (pl.col('A') * 2).alias('D'),
    (pl.col('A') + 10).alias('E')
])
df_pl_mod

A,B,C,D,E
i64,str,bool,i64,i64
1,"""a""",true,2,11
2,"""b""",false,4,12
3,"""c""",true,6,13


### GroupBy Operations

In [30]:
grouped_pd = df_pd.groupby('B').agg({'A': ['sum', 'mean']})
grouped_pd = grouped_pd.reset_index()
grouped_pd

B   A     
     sum mean
0  a   1  1.0
1  b   2  2.0
2  c   3  3.0

#### Polars

In [50]:
df_pl_test = pl.DataFrame({'A': [1, 2, 3], 'B': ['a', 'b', 'c'], 'C': [True, False, True]})
grouped_pl = df_pl_test.group_by('B').agg([
    pl.col('A').sum().alias('A_sum'),
    pl.col('A').mean().alias('A_mean')
])
grouped_pl

B,A_sum,A_mean
str,i64,f64
"""c""",3,3.0
"""b""",2,2.0
"""a""",1,1.0


### Joins

In [31]:
left_pd = pd.DataFrame({'key': ['A', 'B', 'C'], 'value': [1, 2, 3]})
right_pd = pd.DataFrame({'key': ['A', 'B', 'D'], 'other': [4, 5, 6]})
joined_pd = pd.merge(left_pd, right_pd, on='key', how='left')
joined_pd

,key,value,other
0,A,1,4.0
1,B,2,5.0
2,C,3,NaN


#### Polars

In [46]:
left_pl = pl.DataFrame({'key': ['A', 'B', 'C'], 'value': [1, 2, 3]})
right_pl = pl.DataFrame({'key': ['A', 'B', 'D'], 'other': [4, 5, 6]})
joined_pl = left_pl.join(right_pl, on='key', how='left')
joined_pl

key,value,other
str,i64,i64
"""A""",1,4
"""B""",2,5
"""C""",3,null


### Handling Missing Values

In [32]:
df_pd_nulls = pd.DataFrame({'A': [1, None, 3], 'B': [4, 5, None]})
df_pd_filled = df_pd_nulls.fillna(0)
df_pd_filled

,A,B
0,1.0,4.0
1,0.0,5.0
2,3.0,0.0


#### Polars

In [47]:
df_pl_nulls = pl.DataFrame({'A': [1, None, 3], 'B': [4, 5, None]})
df_pl_filled = df_pl_nulls.fill_null(0)
df_pl_filled

A,B
i64,i64
1,4
0,5
3,0


### Sorting

In [33]:
sorted_pd = df_pd.sort_values(by=['A', 'B'], ascending=[True, False])
sorted_pd

,A,B,C,D
0,1,a,True,2
1,2,b,False,4
2,3,c,True,6


#### Polars

In [51]:
df_pl_sort = pl.DataFrame({'A': [3, 1, 2], 'B': ['x', 'y', 'z'], 'C': [True, False, True]})
sorted_pl = df_pl_sort.sort(['A', 'B'], descending=[False, True])
sorted_pl

A,B,C
i64,str,bool
1,"""y""",false
2,"""z""",true
3,"""x""",true


## Expression API: The Heart of the Difference

The most profound difference for day-to-day coding is Polars' expression API versus pandas' more Python-centric approach. Let's explore this with more examples.

### Conditional Logic

In [34]:
import numpy as np
df_pd = pd.DataFrame({'value': [10, 60, 120]})
df_pd['category'] = np.where(df_pd['value'] > 100, 'high', 'low')
df_pd

,value,category
0,10,low
1,60,low
2,120,high


#### Polars: Using when/then/otherwise expressions (simple)

In [36]:
df_pl = pl.DataFrame({'value': [10, 60, 120]})
df_pl = df_pl.with_columns(
    pl.when(pl.col('value') > 100)
      .then(pl.lit('high'))
      .otherwise(pl.lit('low'))
      .alias('category')
)
df_pl

value,category
i64,str
10,"""low"""
60,"""low"""
120,"""high"""


In [35]:
df_pd['category'] = df_pd.apply(
    lambda row: 'high' if row['value'] > 100 else 'medium' if row['value'] > 50 else 'low',
    axis=1
)
df_pd

,value,category
0,10,low
1,60,medium
2,120,high


In [40]:
df_pl = df_pl.with_columns(
    pl.when(pl.col('value') > 100)
      .then(pl.lit('high'))
      .when(pl.col('value') > 50)
      .then(pl.lit('medium'))
      .otherwise(pl.lit('low'))
      .alias('category')
)
df_pl

value,category
i64,str
10,"""low"""
60,"""medium"""
120,"""high"""


#### Polars: More complex conditions chain naturally

### String Operations

In [37]:
df_pd = pd.DataFrame({'name': ['Alice', 'Bob', 'Charlie']})
df_pd['name_upper'] = df_pd['name'].str.upper()
df_pd['contains_a'] = df_pd['name'].str.contains('a')
df_pd

,name,name_upper,contains_a
0,Alice,ALICE,False
1,Bob,BOB,False
2,Charlie,CHARLIE,True


In [41]:
df_pl = pl.DataFrame({'name': ['Alice', 'Bob', 'Charlie']})
df_pl = df_pl.with_columns([
    pl.col('name').str.to_uppercase().alias('name_upper'),
    pl.col('name').str.contains('a').alias('contains_a')
])
df_pl

name,name_upper,contains_a
str,str,bool
"""Alice""","""ALICE""",false
"""Bob""","""BOB""",false
"""Charlie""","""CHARLIE""",true


#### Polars: Using string expressions

### Window Functions

In [38]:
df_pd = pd.DataFrame({'group': ['A', 'A', 'B', 'B'], 'value': [1, 2, 3, 4]})
df_pd['cumsum'] = df_pd.groupby('group')['value'].transform('cumsum')
df_pd['rank'] = df_pd.groupby('group')['value'].rank()
df_pd

,group,value,cumsum,rank
0,A,1,1,1.0
1,A,2,3,2.0
2,B,3,3,1.0
3,B,4,7,2.0


In [42]:
df_pl = pl.DataFrame({'group': ['A', 'A', 'B', 'B'], 'value': [1, 2, 3, 4]})
df_pl = df_pl.with_columns([
    pl.col('value').cum_sum().over('group').alias('cumsum'),
    pl.col('value').rank().over('group').alias('rank')
])
df_pl

group,value,cumsum,rank
str,i64,i64,f64
"""A""",1,1,1.0
"""A""",2,3,2.0
"""B""",3,3,1.0
"""B""",4,7,2.0


#### Polars

## Migration Strategies

Given the differences outlined above, how should you approach migrating from pandas to Polars? Here are some practical strategies:

### 1. Incremental Adoption

You don't need to migrate everything at once. Polars and pandas interoperate well:

In [39]:
df_pl_converted = pl.from_pandas(df_pd)
df_pl_converted

group,value,cumsum,rank
str,i64,i64,f64
"""A""",1,1,1.0
"""A""",2,3,2.0
"""B""",3,3,1.0
"""B""",4,7,2.0


In [43]:
df_pd_back = df_pl.to_pandas()
df_pd_back

,group,value,cumsum,rank
0,A,1,1,1.0
1,A,2,3,2.0
2,B,3,3,1.0
3,B,4,7,2.0


#### Convert Polars DataFrame to pandas

### 2. Start with New Projects

For new data processing projects, consider starting with Polars from the beginning. This avoids the complexity of migration and lets you design around Polars' strengths from the start.

### 3. Focus on Performance Bottlenecks

If you're migrating an existing codebase, focus first on the parts where performance matters most:
- Large dataset operations
- Operations that currently use `apply()`
- Complex aggregations and joins
- Operations in critical processing paths

### 4. Rewrite Common Patterns

Look for these common pandas patterns that should be rewritten in Polars:

- **`df[df['column'] > value]`** → **`df.filter(pl.col('column') > value)`**
- **`df.loc[:, ['A', 'B']]`** → **`df.select(['A', 'B'])`**
- **`df['new_col'] = df['A'] + df['B']`** → **`df = df.with_columns((pl.col('A') + pl.col('B')).alias('new_col'))`**
- **`df.apply(lambda x: ...)`** → Use Polars expressions instead
- **`df.groupby('A').agg(...).reset_index()`** → **`df.group_by('A').agg(...)`**

### 5. Leverage Lazy Evaluation

For complex data pipelines, adopt lazy evaluation to get the full performance benefits:

Instead of eager operations:

```python
df = pl.read_csv("data.csv")
result = (df
    .filter(pl.col('value') > 0)
    .group_by('category')
    .agg(pl.col('value').sum())
)
```

Use lazy evaluation:

```python
result = (pl.scan_csv("data.csv")
    .filter(pl.col('value') > 0)
    .group_by('category')
    .agg(pl.col('value').sum())
    .collect()
)
```

## Conclusion: Is Migration Worth It?

After reviewing the differences and migration strategies, the question remains: is migrating from pandas to Polars worth the effort?

The answer depends on your specific needs, but here are some considerations:

### When Migration Makes Sense

- Your workflows involve large datasets (millions of rows)
- Performance is a bottleneck in your current pandas code
- You're doing complex analytical operations (joins, groupby, window functions)
- You value memory efficiency
- You're starting new projects and can design around Polars from the beginning

### When You Might Want to Stick with pandas

- Your current pandas code performs adequately for your needs
- You rely heavily on pandas' ecosystem integrations
- You have a large codebase that would be costly to migrate
- Your team has deep pandas expertise and limited bandwidth to learn new tools
- You need specialized features that Polars doesn't yet support

Remember that this isn't an all-or-nothing choice. The interoperability between pandas and Polars allows for incremental adoption, using each tool where it makes the most sense.